In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder.appName("DataEngineeringProject").getOrCreate()

employee_data = [
(101, "Sravan", "IT", 75000, "Hyderabad"),
(102, "Ravi", "IT", 68000, "Bangalore"),
(103, "Priya", "Analytics", 62000, "Chennai"),
(104, "Kiran", "HR", 90000, "Mumbai"),
(105, "Anjali", "HR", None, "Pune"),      # Missing salary
(106, "Vikram", "Analytics", 98000, None), # Missing city
(101, "Sravan", "IT", 75000, "Hyderabad")  # Duplicate
]

columns = [
    "emp_id",
    "name",
    "department",
    "salary",
    "city"
]

emp_df = spark.createDataFrame(employee_data, columns)

emp_df.show()

+------+------+----------+------+---------+
|emp_id|  name|department|salary|     city|
+------+------+----------+------+---------+
|   101|Sravan|        IT| 75000|Hyderabad|
|   102|  Ravi|        IT| 68000|Bangalore|
|   103| Priya| Analytics| 62000|  Chennai|
|   104| Kiran|        HR| 90000|   Mumbai|
|   105|Anjali|        HR|  NULL|     Pune|
|   106|Vikram| Analytics| 98000|     NULL|
|   101|Sravan|        IT| 75000|Hyderabad|
+------+------+----------+------+---------+



**Task 1: Remove Duplicate Records**

In [2]:
duplicate_removed = emp_df.dropDuplicates()

print("Duplicate Records Removed")
duplicate_removed.show()

Duplicate Records Removed
+------+------+----------+------+---------+
|emp_id|  name|department|salary|     city|
+------+------+----------+------+---------+
|   102|  Ravi|        IT| 68000|Bangalore|
|   103| Priya| Analytics| 62000|  Chennai|
|   101|Sravan|        IT| 75000|Hyderabad|
|   104| Kiran|        HR| 90000|   Mumbai|
|   106|Vikram| Analytics| 98000|     NULL|
|   105|Anjali|        HR|  NULL|     Pune|
+------+------+----------+------+---------+



**Task 2: Handle Missing Data**

In [3]:
clean_df = duplicate_removed.fillna({
    "salary": 0,
    "city": "Unknown"
})

print("Missing Values Handled")
clean_df.show()

Missing Values Handled
+------+------+----------+------+---------+
|emp_id|  name|department|salary|     city|
+------+------+----------+------+---------+
|   102|  Ravi|        IT| 68000|Bangalore|
|   103| Priya| Analytics| 62000|  Chennai|
|   101|Sravan|        IT| 75000|Hyderabad|
|   104| Kiran|        HR| 90000|   Mumbai|
|   106|Vikram| Analytics| 98000|  Unknown|
|   105|Anjali|        HR|     0|     Pune|
+------+------+----------+------+---------+



**Task 3: Join Multiple Datasets**

In [4]:
manager_data = [
    ("IT", "Ramesh"),
    ("HR", "Suresh"),
    ("Analytics", "Lakshmi")
]

manager_columns = ["department", "manager"]

manager_df = spark.createDataFrame(manager_data, manager_columns)

In [5]:
joined_df = clean_df.join(
    manager_df,
    on="department",
    how="inner"
)

print("Joined Dataset")
joined_df.show()

Joined Dataset
+----------+------+------+------+---------+-------+
|department|emp_id|  name|salary|     city|manager|
+----------+------+------+------+---------+-------+
|        IT|   101|Sravan| 75000|Hyderabad| Ramesh|
|        IT|   102|  Ravi| 68000|Bangalore| Ramesh|
|        HR|   105|Anjali|     0|     Pune| Suresh|
|        HR|   104| Kiran| 90000|   Mumbai| Suresh|
| Analytics|   106|Vikram| 98000|  Unknown|Lakshmi|
| Analytics|   103| Priya| 62000|  Chennai|Lakshmi|
+----------+------+------+------+---------+-------+



**Task 4: Optimize Partitions**

In [6]:
print("Current Partitions:")
print(joined_df.rdd.getNumPartitions())

Current Partitions:
1


In [7]:
optimized_df = joined_df.repartition(2)

print("Partitions After Repartition:")
print(optimized_df.rdd.getNumPartitions())

Partitions After Repartition:
2


**Task 5: Create Aggregated Summary Table**

In [8]:
summary_df = optimized_df.groupBy("department").agg(
    count("*").alias("Total_Employees"),
    avg("salary").alias("Average_Salary"),
    max("salary").alias("Highest_Salary"),
    min("salary").alias("Lowest_Salary")
)

print("Aggregated Summary Table")
summary_df.show()

Aggregated Summary Table
+----------+---------------+--------------+--------------+-------------+
|department|Total_Employees|Average_Salary|Highest_Salary|Lowest_Salary|
+----------+---------------+--------------+--------------+-------------+
|        HR|              2|       45000.0|         90000|            0|
|        IT|              2|       71500.0|         75000|        68000|
| Analytics|              2|       80000.0|         98000|        62000|
+----------+---------------+--------------+--------------+-------------+



**Complete ETL Pipeline**

In [9]:
# ==========================================
# Complete ETL Pipeline
# ==========================================

# Step 1: Extract
# Start with the original employee DataFrame

# Step 2: Transform
# - Remove duplicate employee records
# - Replace missing salary values with 0
# - Replace missing city values with "Unknown"
# - Join employee data with manager data using the department column

etl_df = (
    emp_df
    .dropDuplicates()                         # Remove duplicate rows
    .fillna({
        "salary": 0,                          # Replace null salary with 0
        "city": "Unknown"                     # Replace null city with 'Unknown'
    })
    .join(manager_df, on="department", how="inner")   # Join with manager dataset
)

# Step 3: Load
# Create a department-wise summary report
# Calculate:
# - Total Employees
# - Average Salary
# - Highest Salary
# - Lowest Salary

etl_summary = etl_df.groupBy("department").agg(
    count("*").alias("Employees"),                    # Count employees
    round(avg("salary"), 2).alias("Average_Salary"),  # Average salary
    max("salary").alias("Highest_Salary"),            # Maximum salary
    min("salary").alias("Lowest_Salary")              # Minimum salary
)

# Display the transformed ETL dataset
print("Final ETL Dataset")
etl_df.show()

# Display the aggregated summary report
print("ETL Summary")
etl_summary.show()

Final ETL Dataset
+----------+------+------+------+---------+-------+
|department|emp_id|  name|salary|     city|manager|
+----------+------+------+------+---------+-------+
|        IT|   101|Sravan| 75000|Hyderabad| Ramesh|
|        IT|   102|  Ravi| 68000|Bangalore| Ramesh|
|        HR|   105|Anjali|     0|     Pune| Suresh|
|        HR|   104| Kiran| 90000|   Mumbai| Suresh|
| Analytics|   106|Vikram| 98000|  Unknown|Lakshmi|
| Analytics|   103| Priya| 62000|  Chennai|Lakshmi|
+----------+------+------+------+---------+-------+

ETL Summary
+----------+---------+--------------+--------------+-------------+
|department|Employees|Average_Salary|Highest_Salary|Lowest_Salary|
+----------+---------+--------------+--------------+-------------+
|        IT|        2|       71500.0|         75000|        68000|
|        HR|        2|       45000.0|         90000|            0|
| Analytics|        2|       80000.0|         98000|        62000|
+----------+---------+--------------+----------